# Add Weather to Gold

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    DecimalType,
    TimestampType
)

In [0]:
from pyspark import pipelines as dp

## Variables


##Schema Definition

In [0]:
schema = StructType(
    [
        StructField(
            name="datetime",
            dataType=TimestampType(),
            nullable=False,
            metadata={"comment": "Timestamp of weather observation (hourly)"}
        ),
        StructField(
            name="temperature",
            dataType=DecimalType(6,2),
            nullable=True,
            metadata={"comment": "Temperature at 2 meters above ground in °C"}
        ),
        StructField(
            name="temperature_groups",
            dataType=StringType(),
            nullable=True,
            metadata={"comment": "Groups the temperature in 10°C intervals from -20°C to +30°C"}
        ),
        StructField(
            name="avg_temperature",
            dataType=DecimalType(6,2),
            nullable=True,
            metadata={"comment": "The calculated AVG Temperature of the Day"}
        ),
        StructField(
            name="precipitation",
            dataType=DecimalType(6,2),
            nullable=True,
            metadata={"comment": "Total precipitation in mm"}
        ),
        StructField(
            name="precipitation_groups",
            dataType=StringType(),
            nullable=True,
            metadata={"comment": "Groups the precipitation intervals from 0mm to +50mm"}
        ),
        StructField(
            name="rainy",
            dataType=BooleanType(),
            nullable=True,
            metadata={"comment": "Groups if there was more than 10mm over the whole day"}
        ),
    ]
)

##ETL

In [0]:
@dp.materialized_view(
    # Name der Zieltabelle
    name="analytics.gold.weather",
    # Beschreibung der Tabelle
    comment=f"Weather data for the NYC Area",
    # Liquid Clustering (Statt partitioning und Z-Order)
    cluster_by=[],
    cluster_by_auto=True,
    # Beschreibung des Schemas
    schema=schema,
)
def weather():
    df = spark.read.table(f"analytics.silver.weather_stm_weather")

    # Create Temperature Groups
    df = df.withColumn(
        "temperature_groups",
        F.when(F.col("temperature") < -20, " <-20°C")
        .when(F.col("temperature") < -10, "-20°C - -11°C")
        .when(F.col("temperature") < 0, "-10°C - -1°C")
        .when(F.col("temperature") < 10, "0°C - 9°C")
        .when(F.col("temperature") < 20, "10°C - 19°C")
        .when(F.col("temperature") < 30, "20°C - 29°C")
        .when(F.col("temperature") >= 30, ">=30°C")
        .otherwise(None)
    )

    # Greate AVG Temperature
    df = df.withColumn(
        "avg_temperature",
        F.avg(F.col("temperature")).over(Window.partitionBy(F.col("datetime").cast("date"))))

    # Genarate percipitation groups
    df = df.withColumn(
        "precipitation_groups",
        F.when(F.col("precipitation") < 1, "0")
        .when(F.col("precipitation") < 10, "1-9mm")
        .when(F.col("precipitation") < 20, "10-19mm")
        .when(F.col("precipitation") < 30, "20-29mm")
        .when(F.col("precipitation") < 40, "30-39mm")
        .when(F.col("precipitation") < 50, "40-49mm")
        .when(F.col("precipitation") >= 50, "50+mm")
        .otherwise(None)
    )

    # Get Rainy (based on total daily precipitation)
    df = df.withColumn(
        "daily_precipitation_total",
        F.sum(F.col("precipitation")).over(Window.partitionBy(F.col("datetime").cast("date")))
    )

    df = df.withColumn(
        "rainy",
        F.when(F.col("daily_precipitation_total") > 10, True)
        .otherwise(False)
    ).drop("daily_precipitation_total")


    schema_columns = [(field.name, field.dataType) for field in schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])
    return df